## Fundamentals of quantum walks

A **discrete-time quantum walk** evolves a *walker* over a graph by alternating two operations:

1. a **coin** that mixes the internal degrees of freedom, and
2. a **shift** conditioned on that internal state.

Unlike the diffusive classical walk, interference causes the distribution to split into two *horns* and makes the walker spread **ballistically** (`sigma ~ t`). This notebook walks through the basic examples:

- distribution of the Hadamard coin on the line,
- how the rotation coin tunes the group velocity, and
- animations on the line and on the cycle.

In [ ]:
# Common setup. Run this cell first.
%matplotlib inline
import sys, os
try:
    import zitterwalk 
except ModuleNotFoundError:
    # notebook inside examples/: add the project root to the path
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

from zitterwalk import Graph, Walker, DiscreteTimeWalk
from zitterwalk import viz

### Hadamard walk on the line

Reproduces the two-horn distribution of the DTQW and compares it with the diffusive bell curve of the classical walk. (`examples/line_hadamard.py`)

In [ ]:
n = 201                 # long line so we don't hit the edges
center = n // 2
steps = 80

g = Graph.line(n)
walk = DiscreteTimeWalk(g, coin="hadamard")

# symmetric initial state (|left> + i|right>) / sqrt(2)
w = Walker.at_node(g, center, coin_state=[1, 1j])

states = walk.run(w, steps)
p_final = walk.probabilities(states[-1])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
positions = np.arange(n) - center

# final distribution, keeping only the reachable parity
parity = steps % 2
mask = (np.arange(n) % 2) == (center + parity) % 2
ax1.plot(positions[mask], p_final[mask], color="crimson")
ax1.set_title(f"Hadamard DTQW, t = {steps}")
ax1.set_xlabel("position"); ax1.set_ylabel("probability")

# time evolution (ballistic front)
viz.plot_evolution(walk, states, ax=ax2)

# comparison with classical diffusion (sigma ~ t/sqrt(2))
sigma = steps / np.sqrt(2)
gauss = np.exp(-positions ** 2 / (2 * sigma ** 2))gauss /= gauss.sum()
ax3.plot(positions[mask], p_final[mask], color="crimson", label="quantum")
ax3.plot(positions, gauss, color="navy", ls="--", label="classical (diffusion)")
ax3.set_title("Quantum vs classical"); ax3.set_xlabel("position"); ax3.legend()

fig.tight_layout()

mean = (positions * p_final).sum()
std = np.sqrt(((positions - mean) ** 2 * p_final).sum())
print(f"standard deviation after {steps} steps: {std:.1f}  (~ ballistic, O(t))")

### Tuning the coins

The one-parameter rotation coin `C(theta) = [[cos, sin], [sin, -cos]]` controls how fast the walk spreads: the peaks sit at `±cos(theta)·t`. (`examples/coin_dispersion.py`)

In [ ]:
from zitterwalk import rotation

n = 301
center = n // 2
steps = 100
g = Graph.line(n)
w = Walker.at_node(g, center, coin_state=[1, 1j])
pos = np.arange(n) - center

thetas = [0.15 * np.pi, 0.25 * np.pi, 0.35 * np.pi, 0.45 * np.pi]
fig, ax = plt.subplots(figsize=(9, 5))

for theta in thetas:
    walk = DiscreteTimeWalk(g, coin=rotation(theta))
    final = walk.step(w, steps)
    p = walk.probabilities(final)
    mask = (np.arange(n) % 2) == (center + steps) % 2
    ax.plot(pos[mask], p[mask],
            label=fr"$\theta$ = {theta/np.pi:.2f}$\pi$, "
                  fr"$v = \cos\theta$ = {np.cos(theta):.2f}")
    ax.axvline(np.cos(theta) * steps, color="0.85", lw=0.8, zorder=0)
    ax.axvline(-np.cos(theta) * steps, color="0.85", lw=0.8, zorder=0)

ax.set_xlabel("position"); ax.set_ylabel("probability")
ax.set_title(f"Rotation coin C($\\theta$), t = {steps}\n"
             "(gray lines: predicted peaks at $\\pm\\cos\\theta \\cdot t$)")
ax.legend()
fig.tight_layout()

### Animation on the line

Animates the per-node probability of a Hadamard walk on the line and saves it as a GIF. (`examples/line_animation.py`)

In [ ]:
n = 151
center = n // 2
t_max = 70

g = Graph.line(n)
walk = DiscreteTimeWalk(g, coin="hadamard")
w = Walker.at_node(g, center, coin_state=[1, 1j])
states = walk.run(w, t_max)

viz.animate(walk, states, save_path="line_walk.gif", kind="line", fps=10)
plt.close("all")   # avoid also showing the static frame
Image("line_walk.gif")

### Animation on the cycle

The same style but on a small cycle: the walker goes around and interferes with itself. (`examples/cycle_animation.py`)

In [ ]:
n = 24           # small cycle
start = 0
t_max = 60

g = Graph.cycle(n)
walk = DiscreteTimeWalk(g, coin="hadamard")
w = Walker.at_node(g, start, coin_state=[1, 1j])
states = walk.run(w, t_max)

viz.animate(walk, states, save_path="cycle_walk.gif", kind="line", fps=10)
plt.close("all")
Image("cycle_walk.gif")

### Cycle with colored nodes

The nodes are laid out in a ring and their color is animated according to their probability. (`examples/cycle_nodes_animation.py`)

In [ ]:
n = 24           # small cycle so the nodes are clearly visible
start = 0
t_max = 60

g = Graph.cycle(n)
walk = DiscreteTimeWalk(g, coin="hadamard")
w = Walker.at_node(g, start, coin_state=[1, 1j])
states = walk.run(w, t_max)

viz.animate(walk, states, save_path="cycle_nodes.gif", kind="graph",
            fps=10, node_size=650)
plt.close("all")
Image("cycle_nodes.gif")